In [ ]:
import os
import shutil

print("Setting up dataset...")

# Get the notebook directory (works for all team members)
notebook_dir = os.getcwd()
print(f"Notebook directory: {notebook_dir}")

# Look for dataset folder in Detection and Classification directory
dataset_folder = os.path.join(notebook_dir, "food-1")

# If food-1 exists and has the roboflow.zip but no train folder, copy folders
if os.path.exists(dataset_folder):
    print(f"✓ Found food-1 folder: {dataset_folder}")
    
    # Check if train/valid folders already exist
    train_exists = os.path.exists(os.path.join(dataset_folder, "train"))
    valid_exists = os.path.exists(os.path.join(dataset_folder, "valid"))
    
    if train_exists and valid_exists:
        print("✓ train/ and valid/ folders already exist in food-1")
    else:
        # Look for source folders to copy from
        print("\n→ Copying train/valid folders to food-1...")
        
        # Check in parent directory for IndianFoodNet
        parent_dir = os.path.dirname(notebook_dir)
        source_dataset = os.path.join(parent_dir, "IndianFoodNet.v1i.yolov8 (Unzipped Files)")
        
        if not os.path.exists(source_dataset):
            source_dataset = os.path.join(parent_dir, "IndianFoodNet.v1i.yolov8")
        
        if os.path.exists(source_dataset):
            print(f"  Found source at: {source_dataset}")
            
            # Copy train folder
            src_train = os.path.join(source_dataset, "train")
            dst_train = os.path.join(dataset_folder, "train")
            
            if os.path.exists(src_train) and not os.path.exists(dst_train):
                print(f"  → Copying train folder...")
                shutil.copytree(src_train, dst_train)
                print(f"  ✓ train/ copied to food-1")
            
            # Copy valid folder
            src_valid = os.path.join(source_dataset, "valid")
            dst_valid = os.path.join(dataset_folder, "valid")
            
            if os.path.exists(src_valid) and not os.path.exists(dst_valid):
                print(f"  → Copying valid folder...")
                shutil.copytree(src_valid, dst_valid)
                print(f"  ✓ valid/ copied to food-1")
        else:
            print(f"✗ Could not find IndianFoodNet source folder")

# Now set up dataset object
class DatasetPath:
    def __init__(self, path):
        self.location = path

if os.path.exists(dataset_folder):
    dataset = DatasetPath(dataset_folder)
    
    train_path = os.path.join(dataset.location, "train")
    valid_path = os.path.join(dataset.location, "valid")
    
    if os.path.exists(train_path):
        print(f"✓ Train folder: OK")
    else:
        print(f"✗ Train folder: NOT FOUND")
    
    if os.path.exists(valid_path):
        print(f"✓ Valid folder: OK")
    else:
        print(f"✗ Valid folder: NOT FOUND")
    
    print(f"\n✓ Dataset ready at: ./food-1/")
else:
    print(f"✗ food-1 folder not found in {notebook_dir}")
    dataset = None

In [ ]:
dataset

In [ ]:
import glob

train_path = dataset.location + "/train"

images = glob.glob(train_path + "/**/*.jpg", recursive=True)

print(len(images))

In [ ]:
import os
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image

# Get dataset path
data_path = dataset.location  # from Roboflow download

# Collect all training images (recursive search)
image_paths = glob.glob(os.path.join(data_path, "train", "**", "*.jpg"), recursive=True)
image_paths += glob.glob(os.path.join(data_path, "train", "**", "*.png"), recursive=True)

In [ ]:
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

# Verify dataset is available
if dataset is None:
    print("✗ ERROR: dataset variable not set. Please run cell 2 first and ensure it finds the dataset.")
else:
    # Get dataset path
    data_path = dataset.location
    print(f"Using dataset path: {data_path}")
    
    # Check if data path exists
    if not os.path.exists(data_path):
        print(f"✗ ERROR: Dataset path does not exist: {data_path}")
    else:
        # Check for CSV-based format (Roboflow)
        csv_path = os.path.join(data_path, "train", "_classes.csv")
        
        if os.path.exists(csv_path):
            print(f"✓ Found CSV-based dataset (Roboflow format)")
            print(f"✓ CSV loaded from: {csv_path}")
            
            try:
                # load CSV
                df = pd.read_csv(csv_path)
                print(f"✓ CSV loaded successfully! Shape: {df.shape}")
                
                # pick 4 random rows
                sample = df.sample(4)
                
                plt.figure(figsize=(6, 6))
                
                for i, row in enumerate(sample.itertuples(index=False)):
                    filename = row[0]  # first column = filename
                    
                    # find class (column where value = 1)
                    class_idx = list(row[1:]).index(1)
                    class_name = df.columns[1:][class_idx]
                    
                    # load image
                    img_path = os.path.join(data_path, "train", filename)
                    
                    if not os.path.exists(img_path):
                        print(f"✗ Warning: Image not found: {img_path}")
                        continue
                    
                    img = Image.open(img_path)
                    
                    plt.subplot(2, 2, i+1)
                    plt.imshow(img)
                    plt.title(class_name)
                    plt.axis("off")
                
                plt.tight_layout()
                plt.show()
                print("✓ Images displayed successfully!")
                
            except Exception as e:
                print(f"✗ Error processing CSV or images: {e}")
        
        else:
            # Check for YOLO format (images and txt label files)
            train_images_path = os.path.join(data_path, "train", "images")
            if os.path.exists(train_images_path):
                print(f"✓ Found YOLO-based dataset (with images folder)")
                print(f"✓ Image folder at: {train_images_path}")
                
                # Get all image files
                image_files = []
                for ext in ["*.jpg", "*.jpeg", "*.png"]:
                    image_files.extend([f for f in os.listdir(train_images_path) if f.lower().endswith(ext.replace("*", ""))])
                
                if image_files:
                    print(f"✓ Found {len(image_files)} images in training set")
                    
                    # Display 4 random images
                    sample_images = random.sample(image_files, min(4, len(image_files)))
                    
                    plt.figure(figsize=(8, 8))
                    
                    for i, img_file in enumerate(sample_images):
                        img_path = os.path.join(train_images_path, img_file)
                        img = Image.open(img_path)
                        
                        plt.subplot(2, 2, i+1)
                        plt.imshow(img)
                        plt.title(img_file[:20])  # Truncate filename
                        plt.axis("off")
                    
                    plt.tight_layout()
                    plt.show()
                    print("✓ Images displayed successfully!")
                else:
                    print(f"✗ No image files found in {train_images_path}")
            else:
                print(f"✗ ERROR: Dataset format not recognized")
                print(f"✓ Available folders in {data_path}:")
                print(os.listdir(data_path))
                print(f"\nExpected either:")
                print(f"  - CSV format: {data_path}/train/_classes.csv")
                print(f"  - YOLO format: {data_path}/train/images/")

In [ ]:
import torchvision.transforms as transforms
import torchvision.transforms.functional as F

class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)
        hp = (max_wh - w) // 2
        vp = (max_wh - h) // 2
        padding = (hp, vp, max_wh - w - hp, max_wh - h - vp)
        return F.pad(image, padding, 0, "constant")

transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((224, 224)),  # required for pretrained models
    transforms.ToTensor(),
])

In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image
import torch
import glob

class FoodDataset(Dataset):
    """
    Supports both YOLO format (images/ and labels/) and CSV format datasets
    """
    def __init__(self, root, split="train", transform=None):
        self.root = os.path.join(root, split)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_names = []
        
        # Try YOLO format first (images/ and labels/)
        images_dir = os.path.join(self.root, "images")
        labels_dir = os.path.join(self.root, "labels")
        
        if os.path.exists(images_dir) and os.path.exists(labels_dir):
            # YOLO format dataset
            self._load_yolo_format(images_dir, labels_dir)
        else:
            # Try CSV format (Roboflow)
            self._load_csv_format()
    
    def _load_yolo_format(self, images_dir, labels_dir):
        """Load YOLO format dataset with images/ and labels/ folders"""
        import glob
        
        # Get all image files
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
        image_files = []
        for ext in image_extensions:
            image_files.extend(glob.glob(os.path.join(images_dir, ext)))
        
        image_files = sorted(image_files)
        
        # Collect unique class names from all label files
        class_set = set()
        
        for img_path in image_files:
            img_name = os.path.basename(img_path)
            label_name = os.path.splitext(img_name)[0] + '.txt'
            label_path = os.path.join(labels_dir, label_name)
            
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    content = f.read().strip()
                    if content:
                        # YOLO format: class_id x_center y_center width height (for detection)
                        # For classification: just class_id
                        class_id = int(content.split()[0])
                        class_set.add(class_id)
                        self.image_paths.append(img_path)
                        self.labels.append(class_id)
        
        # Create class names (0, 1, 2, ... or from data.yaml if available)
        num_classes = max(self.labels) + 1 if self.labels else 0
        self.class_names = [f"class_{i}" for i in range(num_classes)]
        
        print(f"✓ Loaded YOLO format: {len(self.image_paths)} images, {len(self.class_names)} classes")
    
    def _load_csv_format(self):
        """Load CSV format dataset (Roboflow)"""
        import pandas as pd
        
        csv_path = os.path.join(self.root, "_classes.csv")
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"No CSV or YOLO format found in {self.root}")
        
        df = pd.read_csv(csv_path)
        self.class_names = list(df.columns[1:])
        
        for idx, row in df.iterrows():
            filename = row[0]
            img_path = os.path.join(self.root, filename)
            label_vec = row[1:].astype(int).values
            label = int(label_vec.argmax())
            
            if os.path.exists(img_path):
                self.image_paths.append(img_path)
                self.labels.append(label)
        
        print(f"✓ Loaded CSV format: {len(self.image_paths)} images, {len(self.class_names)} classes")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
from torch.utils.data import DataLoader

train_ds = FoodDataset(dataset.location, "train", transform)
val_ds = FoodDataset(dataset.location, "valid", transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
import torchvision.models as models

# Redefine transforms with augmentation for training
train_transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.RandomPerspective(distortion_scale=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Create data loaders with augmented transforms
train_ds_aug = FoodDataset(dataset.location, "train", train_transform)
val_ds_aug = FoodDataset(dataset.location, "valid", val_transform)

train_loader_aug = DataLoader(train_ds_aug, batch_size=32, shuffle=True)
val_loader_aug = DataLoader(val_ds_aug, batch_size=32)

num_classes = len(train_ds_aug.class_names)

# Model (ResNet-18)
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, num_classes)
)

# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Live accuracy plot
train_acc_history = []
val_acc_history = []

plt.ion()
fig, ax = plt.subplots()

# Training loop
epochs = 10

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total


    # ---- VALIDATION ----
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader_aug:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total


    # ---- Plot update ----
    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    ax.clear()
    ax.plot(train_acc_history, label="Train Accuracy")
    ax.plot(val_acc_history, label="Validation Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
    fig.canvas.draw()
    fig.canvas.flush_events()
    plt.pause(0.01)

    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")


plt.ioff()
plt.show()

In [ ]:
# Try to load previously trained weights
model_save_path = os.path.join(dataset.location, "resnet18_trained.pth")

if os.path.exists(model_save_path):
    model.load_state_dict(torch.load(model_save_path, map_location=device))
    print(f"✓ Loaded previously trained model from: {model_save_path}")
    print("⚠ Model loaded! You can skip training or retrain to improve further.")
else:
    print("ℹ No saved model found. Proceed to train the model.")

In [ ]:
# Save the trained model
import os

model_save_path = os.path.join(dataset.location, "resnet18_trained.pth")
torch.save(model.state_dict(), model_save_path)
print(f"✓ Model saved to: {model_save_path}")
print(f"✓ File size: {os.path.getsize(model_save_path) / (1024**2):.2f} MB")